# HUGGING FACES LLM COURSES CHAPTER 2 : **USİNG** **TRANSFORMERS**

## 2 ) BEHİND THE PİPELİNE

In [ ]:
#KOD1: BEHİND THE PİPELİNE
#-------------------------------------------------------------------

from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [ ]:
#KOD2: BEHİND THE PİPELİNE
#-------------------------------------------------------------------

from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
#KOD3: BEHİND THE PİPELİNE
#-------------------------------------------------------------------

raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [ ]:
#KOD4: BEHİND THE PİPELİNE
#-------------------------------------------------------------------

from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([2, 16, 768])


In [ ]:
#KOD5: BEHİND THE PİPELİNE
#-------------------------------------------------------------------

from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

NameError: name 'inputs' is not defined

In [ ]:
#KOD6: BEHİND THE PİPELİNE
#-------------------------------------------------------------------

print(outputs.logits.shape)

torch.Size([2, 2])


In [ ]:
#KOD7: BEHİND THE PİPELİNE
#-------------------------------------------------------------------

print(outputs.logits)

tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)


In [ ]:
#KOD8: BEHİND THE PİPELİNE
#-------------------------------------------------------------------

import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


In [ ]:
#KOD9: BEHİND THE PİPELİNE
#-------------------------------------------------------------------

model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

In [ ]:
#KOD10: GEÇMİŞ UYGULAMALARIN ÖRNEĞİ

# ===== SAYFA 2: PIPELINE'IN ARKASINDA NE OLUYOR? =====

# 1) Gerekli kütüphaneleri getir
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 2) Kullanacağımız yapay zekânın adı (pipeline da arkada bunu kullanıyor)
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

# 3) Çevirmeni (tokenizer) ve modeli indir
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

# 4) Analiz edeceğimiz cümleler (istersen kendi cümlelerini yaz türkçeyi de yap)
raw_inputs = [
    "Mesut POLATGİL is a highly successful software developer in her field.",
    "That is why we are very proud",
]

# ADIM 1: TOKENIZER -> Yazıyı sayıya çevir
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print("1) Cümlelerin sayı hali:\n", inputs["input_ids"])

# ADIM 2: MODEL -> Sayılardan ham puan (logits) üret
outputs = model(**inputs)
print("\n2) Modelin ham puanları (logits):\n", outputs.logits)

# ADIM 3: SOFTMAX -> Ham puanı yüzdeye (olasılığa) çevir
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print("\n3) Olasılıklar:\n", predictions)

# ADIM 4: ETİKET -> En yüksek olasılığın hangi sınıf olduğunu bul
print("\n4) Sonuç:")
for cumle, olasilik in zip(raw_inputs, predictions):
    en_yuksek = olasilik.argmax().item()  # en büyük puanın sırası (0 veya 1)
    etiket = model.config.id2label[en_yuksek]  # 0 -> NEGATIVE, 1 -> POSITIVE
    print(f"  '{cumle}' -> {etiket} (%{olasilik[en_yuksek].item() * 100:.1f})")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

1) Cümlelerin sayı hali:
 tensor([[  101,  2033,  6342,  2102, 14955,  4017, 20142,  2003,  1037,  3811,
          3144,  4007,  9722,  1999,  2014,  2492,  1012,   102],
        [  101,  2008,  2003,  2339,  2057,  2024,  2200,  7098,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0]])

2) Modelin ham puanları (logits):
 tensor([[-3.7474,  3.9850],
        [-4.2972,  4.6683]], grad_fn=<AddmmBackward0>)

3) Olasılıklar:
 tensor([[4.3821e-04, 9.9956e-01],
        [1.2772e-04, 9.9987e-01]], grad_fn=<SoftmaxBackward0>)

4) Sonuç:
  'Mesut POLATGİL is a highly successful software developer in her field.' -> POSITIVE (%100.0)
  'That is why we are very proud' -> POSITIVE (%100.0)


# 3) MODELS

In [ ]:
#KOD11: MODELS
#-------------------------------------------------------------------

from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#KOD12: MODELS
#-------------------------------------------------------------------

from transformers import BertModel

model = BertModel.from_pretrained("bert-base-cased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#KOD13: MODELS
#-------------------------------------------------------------------

model.save_pretrained("directory_on_my_computer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#KOD14: MODELS
#-------------------------------------------------------------------

!ls directory_on_my_computer

config.json  model.safetensors


In [ ]:
#KOD15: MODELS
#-------------------------------------------------------------------

from transformers import AutoModel

model = AutoModel.from_pretrained("directory_on_my_computer")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
#KOD16: MODELS
#-------------------------------------------------------------------

from huggingface_hub import notebook_login

notebook_login()

In [ ]:
#KOD17: MODELS
#-------------------------------------------------------------------

model.push_to_hub("my-awesome-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nq7wzun/model.safetensors:   4%|3         | 15.9MB /  433MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/gamzeay/my-awesome-model/commit/490e00467e4a7717d7dfaeb30a04018916676fef', commit_message='Upload model', commit_description='', oid='490e00467e4a7717d7dfaeb30a04018916676fef', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gamzeay/my-awesome-model', endpoint='https://huggingface.co', repo_type='model', repo_id='gamzeay/my-awesome-model'), pr_revision=None, pr_num=None)

In [ ]:
#KOD18: MODELS
#-------------------------------------------------------------------

from transformers import AutoModel

model = AutoModel.from_pretrained("gamzeay/my-awesome-model")

config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
#KOD19: MODELS
#-------------------------------------------------------------------

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

encoded_input = tokenizer("Hello, I'm a single sentence!")
print(encoded_input)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

{'input_ids': [101, 8667, 117, 146, 112, 182, 170, 1423, 5650, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
#KOD20: MODELS
#-------------------------------------------------------------------

tokenizer.decode(encoded_input["input_ids"])

"[CLS] Hello, I ' m a single sentence! [SEP]"

In [ ]:
#KOD21: MODELS
#-------------------------------------------------------------------

encoded_input = tokenizer("How are you?", "I'm fine, thank you!")
print(encoded_input)

{'input_ids': [101, 1731, 1132, 1128, 136, 102, 146, 112, 182, 2503, 117, 6243, 1128, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
#KOD22: MODELS
#-------------------------------------------------------------------

encoded_input = tokenizer(["How are you?", "I'm fine, thank you!"])
print(encoded_input)

{'input_ids': [[101, 1731, 1132, 1128, 136, 102], [101, 146, 112, 182, 2503, 117, 6243, 1128, 106, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}


In [ ]:
#KOD23: MODELS
#-------------------------------------------------------------------

encoded_input = tokenizer("How are you?", "I'm fine, thank you!", return_tensors="pt")
print(encoded_input)

{'input_ids': tensor([[ 101, 1731, 1132, 1128,  136,  102,  146,  112,  182, 2503,  117, 6243,
         1128,  106,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
#KOD24: MODELS claudenin hücre 13 dediği kod Hücre 13: Padding (dolgu) ile sorunu çöz
#-------------------------------------------------------------------

encoded_input = tokenizer(
    ["How are you?", "I'm fine, thank you!"], padding=True, return_tensors="pt"
)
print(encoded_input)

{'input_ids': tensor([[ 101, 1731, 1132, 1128,  136,  102,    0,    0,    0,    0],
        [ 101,  146,  112,  182, 2503,  117, 6243, 1128,  106,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
#KOD25: MODELS Hücre 14: Çok uzun cümleyi kesme (truncation) python
#-------------------------------------------------------------------
#Dökümandan küçük bir fark: Dökümanda cümle elle yazılmış, içinde yaklaşık 50 tane “very” var.
#Bu kadarı sınırı aşmadığı için kesilme olmaz, dökümandaki uzun çıktı da bu cümleyle çıkmaz.
#Ben "very " * 600 ile aynı kelimeyi 600 kez tekrarlattım, böylece kesilme gerçekten görünüyor.
#Ne yapıyor?
#truncation=True → “Model için çok uzunsa kes” demek.
#BERT en fazla 512 token işleyebilir, bu yüzden 512'den sonrasını atıyor.
#Önemli olan bir şey var: dökümandaki örnekte listenin sonunda “long sentence.” kelimelerinin numaraları (1179, 5650, 119) vardı.
#Senin çıktında yok, çünkü cümle çok uzundu ve sondaki “long sentence.” kısmı kesilip atıldı. Truncation’ın çalıştığının kanıtı bu.
#Listenin sonuna 102 işaretini tokenizer ekledi.
#Sunumda şöyle anlatabilirsin: “Cümlenin başı duruyor, sonu kesildi, model en fazla 512 token alabiliyor.”

encoded_input = tokenizer(
    "This is a " + "very " * 600 + "long sentence.",
    truncation=True,
)
print(len(encoded_input["input_ids"]))
print(encoded_input["input_ids"])

512
[101, 1188, 1110, 170, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1304, 1

In [ ]:
#KOD26: MODELS
#-------------------------------------------------------------------

encoded_input = tokenizer(
    ["How are you?", "I'm fine, thank you!"],
    padding=True,
    truncation=True,
    max_length=5,
    return_tensors="pt",
)
print(encoded_input)

{'input_ids': tensor([[ 101, 1731, 1132, 1128,  102],
        [ 101,  146,  112,  182,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1]])}


In [ ]:
#KOD27: MODELS Hücre 16: Özel işaretler (special tokens)
#-------------------------------------------------------------------

encoded_input = tokenizer("How are you?")
print(encoded_input["input_ids"])
tokenizer.decode(encoded_input["input_ids"])

[101, 1731, 1132, 1128, 136, 102]


'[CLS] How are you? [SEP]'

In [ ]:
#KOD28:MODELS
#-------------------------------------------------------------------
#Çıktı yok. Bu bizim bildiğimiz iki cümle.

sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]

In [ ]:
#KOD29:MODELS
#-------------------------------------------------------------------
#Çıktı yok. Bunlar Sayfa 2'de tokenizer’ın ürettiği sayılar, burada elle yazılmış hali.

encoded_sequences = [
    [
        101,
        1045,
        1005,
        2310,
        2042,
        3403,
        2005,
        1037,
        17662,
        12172,
        2607,
        2026,
        2878,
        2166,
        1012,
        102,
    ],
    [101, 1045, 5223, 2023, 2061, 2172, 999, 102],
]

In [ ]:
#KOD30:MODELS Hücre 19: Tabloya çevir ⚠️ (bu hücre hata verecek)
#-------------------------------------------------------------------
#Dökümanın söylediği: “Bu liste zaten dikdörtgen, kolayca tabloya çevrilir.”
#Gerçek durum: Dökümanda bir yanlışlık var. Birinci satırda 16 sayı, ikincide 8 sayı var, yani dikdörtgen değil.
#Bu yüzden şuna benzer bir hata alacaksın:
#Hata mesajının tam kelimeleri sürüme göre değişebilir.
#Bu hata sunumda anlatabileceğin iyi bir örnek: “Tablolar dikdörtgen olmak zorunda, bu yüzden padding gerekiyor.”

import torch

model_inputs = torch.tensor(encoded_sequences)

ValueError: expected sequence of length 16 at dim 1 (got 8)

In [ ]:
#KOD30:MODELS Hücre 20: Hatayı çöz (dolgu ekle)
#-------------------------------------------------------------------
#İkinci cümlenin sonuna 8 tane 0 (dolgu) ekliyoruz. Böylece ikisi de 16 uzunlukta oluyor.
#Sonra tabloya çeviriyoruz.


encoded_sequences[1] = encoded_sequences[1] + [0] * 8

model_inputs = torch.tensor(encoded_sequences)
print(model_inputs.shape)

torch.Size([2, 16])


In [ ]:
#KOD30:MODELS  Hücre 21: Tabloyu modele ver
#-------------------------------------------------------------------

output = model(model_inputs)
print(output.last_hidden_state.shape)

torch.Size([2, 16, 768])


4) TOKENİZERS


In [ ]:
#KOD30:TOKENİZERS Hücre 22: Kelime bazlı bölme
#-------------------------------------------------------------------

tokenized_text = "Jim Henson was a puppeteer".split()
print(tokenized_text)

['Jim', 'Henson', 'was', 'a', 'puppeteer']


In [ ]:
#KOD31:TOKENİZERS Hücre 23: Tokenizer’ı iki yolla yükle YOL 1
#-------------------------------------------------------------------
#BertTokenizer → BERT’e özel sınıfı elle seçiyorsun.

from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
#KOD31:TOKENİZERS Hücre 23: Tokenizer’ı iki yolla yükle YOL 2
#-------------------------------------------------------------------
#AutoTokenizer → modelin adına bakıp doğru olanı otomatik seçiyor. OTOMATİK YAPMAK DAHA PRATİKTİR.

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
#KOD32:TOKENİZERS Hücre 24: Tokenizer’ı doğrudan kullan
#-------------------------------------------------------------------

tokenizer("Using a Transformer network is simple")

{'input_ids': [101, 7993, 170, 13809, 23763, 2443, 1110, 3014, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
#KOD33:TOKENİZERS Hücre 25: Tokenizer’ı kaydet
#-------------------------------------------------------------------

tokenizer.save_pretrained("directory_on_my_computer")

('directory_on_my_computer/tokenizer_config.json',
 'directory_on_my_computer/tokenizer.json')

In [ ]:
#KOD34:TOKENİZERS Hücre 26: Adım 1, parçalara böl
#-------------------------------------------------------------------

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)

print(tokens)

['Using', 'a', 'Trans', '##former', 'network', 'is', 'simple']


In [ ]:
#KOD35:TOKENİZERS Hücre 27: Adım 2, parçaları sayıya çevir
#-------------------------------------------------------------------

ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)

[7993, 170, 13809, 23763, 2443, 1110, 3014]


In [ ]:
#KOD36:TOKENİZERS Hücre 28: Decode (sayıdan yazıya geri dönü
#-------------------------------------------------------------------

decoded_string = tokenizer.decode([7993, 170, 11303, 1200, 2443, 1110, 3014])
print(decoded_string)

Using a transformer network is simple


In [ ]:
#KOD37:TOKENİZERS Hücre 29: Dökümanın “Try it out” ödevi
#-------------------------------------------------------------------

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

for sentence in [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]:
    tokens = tokenizer.tokenize(sentence)
    ids = tokenizer.convert_tokens_to_ids(tokens)
    print(ids)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012]
[1045, 5223, 2023, 2061, 2172, 999]


5) HANDLİNG MULTİPLE SEQUENCES

In [ ]:
#KOD38:HANDLİNG MULTİPLE SEQUENCES Hücre 30: Bilerek hata alacağımız kod ⚠️
#-------------------------------------------------------------------
#Sınıfa şöyle anlat: “Model tek cümleyi değil, cümle listesini bekliyor.
#Tokenizer’ı doğrudan çağırdığımızda bu işi bizim yerimize yapıyordu.”

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)
# This line will fail.
model(input_ids)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

RuntimeError: The size of tensor a (14) must match the size of tensor b (512) at non-singleton dimension 1

In [ ]:
#KOD39:HANDLİNG MULTİPLE SEQUENCES Hücre 31: Tokenizer’ın eklediği “tepsiyi” gör
#-------------------------------------------------------------------

tokenized_inputs = tokenizer(sequence, return_tensors="pt")
print(tokenized_inputs["input_ids"])

tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102]])


In [ ]:
#KOD40:HANDLİNG MULTİPLE SEQUENCES Hücre 32: Tepsiyi biz ekleyelim (hata düzeliyor)
#-------------------------------------------------------------------

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor([ids])
print("Input IDs:", input_ids)

output = model(input_ids)
print("Logits:", output.logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


In [ ]:
#KOD41:HANDLİNG MULTİPLE SEQUENCES  Hücre 33: Batch (toplu cümle) nedir? Aynı cümleyi iki kere ver
#-------------------------------------------------------------------

batched_ids = [ids, ids]

#Bu, iki aynı cümlelik bir tepsi. Ardından ödevin istediği: bunu tabloya çevirip modele ver ve logits’in iki kere aynı çıktığını gör:

In [ ]:
#KOD41:HANDLİNG MULTİPLE SEQUENCES  - 41 DEVAM KODU - Try it out ÖDEVİ ASLINDA BU
#-------------------------------------------------------------------

input_ids = torch.tensor(batched_ids)
output = model(input_ids)
print(output.logits)

#Anlamı: İki aynı cümle verdik, iki aynı cevap aldık. Batching (toplu işleme) bu demek: birden fazla cümleyi tek seferde modele vermek. Tek cümle de verebiliriz, onu tek elemanlı bir tepsiye koyarız.
#Parça A’nın özeti (sunum için)
#Model cümle listesi bekler.
#Tek cümleyi doğrudan verirsen hata alırsın.
#Tek cümle bile olsa bir köşeli parantezle sarıp “liste” yaparsın.
#Tokenizer’ı doğrudan çağırırsan bu sarmayı o kendisi yapar.

tensor([[-2.7276,  2.8789],
        [-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


In [ ]:
#KOD42:HANDLİNG MULTİPLE SEQUENCES Hücre 34: Dikdörtgen olmayan tablo ⚠️ (bilerek hata alacağız)
#-------------------------------------------------------------------
#Ne yapıyor? Birinci satırda 3, ikincide 2 sayı var, tablo dikdörtgen değil.
#(Buradaki 200 sayıları anlamsız örnek sayılar, bir kelimeye karşılık gelmesi önemli değil.)

import torch

batched_ids = [
    [200, 200, 200],
    [200, 200],
]
torch.tensor(batched_ids)

ValueError: expected sequence of length 3 at dim 1 (got 2)

In [ ]:
#KOD43:HANDLİNG MULTİPLE SEQUENCES Hücre 35: Dolgu ekleyerek çöz
#-------------------------------------------------------------------

padding_id = 100

batched_ids = [
    [200, 200, 200],
    [200, 200, padding_id],
]

In [ ]:
#KOD44:HANDLİNG MULTİPLE SEQUENCES  Hücre 36: Dolgunun sonucu bozduğunu gör
#-------------------------------------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


In [ ]:
#KOD45:HANDLİNG MULTİPLE SEQUENCES Hücre 37: Attention mask ile çözüm
#-------------------------------------------------------------------

batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

attention_mask = [
    [1, 1, 1],
    [1, 1, 0],
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


In [ ]:
#KOD46:HANDLİNG MULTİPLE SEQUENCES Hücre 38: Dökümandaki “Try it out” ödevi
#-------------------------------------------------------------------

sentence1 = "I've been waiting for a HuggingFace course my whole life."
sentence2 = "I hate this so much!"

# Tokenizer'ı doğrudan çağırıyoruz, böylece başta 101, sonda 102 de ekleniyor
ids1 = tokenizer(sentence1)["input_ids"]
ids2 = tokenizer(sentence2)["input_ids"]

# 1) Cümleler tek tek
print("1. cümle:", model(torch.tensor([ids1])).logits)
print("2. cümle:", model(torch.tensor([ids2])).logits)

# 2) Dolgu ekle (kısa cümleyi uzun olanla eşitle)
fark = len(ids1) - len(ids2)
pad = tokenizer.pad_token_id
batched_ids = [ids1, ids2 + [pad] * fark]

# 3) Maskesiz ver (dolgu sonucu bozacak)
print("Maskesiz:", model(torch.tensor(batched_ids)).logits)

# 4) Maskeli ver (düzelecek)
attention_mask = [[1] * len(ids1), [1] * len(ids2) + [0] * fark]
print("Maskeli:", model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask)).logits)

1. cümle: tensor([[-1.5607,  1.6123]], grad_fn=<AddmmBackward0>)
2. cümle: tensor([[ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)
Maskesiz: tensor([[-1.5607,  1.6123],
        [ 3.4603, -2.9194]], grad_fn=<AddmmBackward0>)
Maskeli: tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)


In [ ]:
#KOD47:HANDLİNG MULTİPLE SEQUENCES Hücre 39: Uzun cümleler ⚠️ (dökümandaki kodda düzeltme var)
#-------------------------------------------------------------------

sequence = sequence[:max_sequence_length]

NameError: name 'max_sequence_length' is not defined

In [ ]:
#KOD48:HANDLİNG MULTİPLE SEQUENCES
#-------------------------------------------------------------------

max_sequence_length = 20

sequence = "I've been waiting for a HuggingFace course my whole life."
sequence = sequence[:max_sequence_length]
print(sequence)

#Sayfa 5'in özeti (sunum için)
#Model cümle listesi (batch) bekler.
#Cümleler farklı uzunlukta olursa padding ile eşitlenir.
#Dolgu sonucu bozar, o yüzden attention mask ile modele “dolguya bakma” denir.
#Çok uzun cümleler truncation ile kesilir.

I've been waiting fo


6) PUTTING IN ALL TOGETHER

In [ ]:
#KOD49:PUTTING IN ALL TOGETHER  Hücre 40: Tokenizer’ı yükle ve tek cümle ver
#-------------------------------------------------------------------

from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

model_inputs = tokenizer(sequence)
print(model_inputs)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

{'input_ids': [101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
#KOD50:PUTTING IN ALL TOGETH Hücre 41: Birden fazla cümle (API değişmiyor)
#-------------------------------------------------------------------

sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

model_inputs = tokenizer(sequences)
print(model_inputs)

{'input_ids': [[101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102], [101, 2061, 2031, 1045, 999, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}


In [ ]:
#KOD51:PUTTING IN ALL TOGETH Hücre 42: Dolgu (padding) çeşitleri
#-------------------------------------------------------------------

# Cümleleri en uzun cümlenin boyuna kadar doldur
model_inputs = tokenizer(sequences, padding="longest")
print([len(x) for x in model_inputs["input_ids"]])

# Cümleleri modelin sınırına kadar doldur (BERT ve DistilBERT için 512)
model_inputs = tokenizer(sequences, padding="max_length")
print([len(x) for x in model_inputs["input_ids"]])

# Cümleleri senin belirlediğin uzunluğa kadar doldur
model_inputs = tokenizer(sequences, padding="max_length", max_length=8)
print([len(x) for x in model_inputs["input_ids"]])

[16, 16]
[512, 512]
[16, 8]


In [ ]:
#KOD52:PUTTING IN ALL TOGETH Hücre 43: Kesme (truncation) çeşitleri
#-------------------------------------------------------------------

# Modelin sınırından uzun olanları kes (BERT ve DistilBERT için 512)
model_inputs = tokenizer(sequences, truncation=True)
print([len(x) for x in model_inputs["input_ids"]])

# Belirlediğin uzunluktan uzun olanları kes
model_inputs = tokenizer(sequences, max_length=8, truncation=True)
print([len(x) for x in model_inputs["input_ids"]])

[16, 6]
[8, 6]


In [ ]:
#KOD53:PUTTING IN ALL TOGETH Hücre 44: Framework’e uygun çıktı (PyTorch ve NumPy)
#-------------------------------------------------------------------

# PyTorch tablosu (tensor) döndürür
model_inputs = tokenizer(sequences, padding=True, return_tensors="pt")
print(model_inputs["input_ids"].shape)

# NumPy dizisi döndürür
model_inputs = tokenizer(sequences, padding=True, return_tensors="np")
print(model_inputs["input_ids"].shape)

torch.Size([2, 16])
(2, 16)


In [ ]:
#KOD54:PUTTING IN ALL TOGETH Hücre 45: Özel işaretler
#-------------------------------------------------------------------

sequence = "I've been waiting for a HuggingFace course my whole life."

model_inputs = tokenizer(sequence)
print(model_inputs["input_ids"])

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[101, 1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012, 102]
[1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012]


In [ ]:
#KOD55:PUTTING IN ALL TOGETH Hücre 46: Bu işaretler neymiş, geri çevirerek gör
#-------------------------------------------------------------------

print(tokenizer.decode(model_inputs["input_ids"]))
print(tokenizer.decode(ids))

[CLS] i ' ve been waiting for a huggingface course my whole life. [SEP]
i ' ve been waiting for a huggingface course my whole life.


In [ ]:
#KOD56:PUTTING IN ALL TOGETH Hücre 47: Hepsini birleştir, tokenizer’dan modele
#-------------------------------------------------------------------

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

tokens = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")
output = model(**tokens)
print(output.logits)

#Sayfa 6'nın özeti (sunum için)
#Tokenizer tek komutla hepsini yapar:
#Tek ya da çok cümle alır
#Padding ve truncation uygular
#Özel işaretleri ekler
#Çıktıyı istediğin formatta verir ("pt", "np")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[-1.5607,  1.6123],
        [-3.6183,  3.9137]], grad_fn=<AddmmBackward0>)
